# GnuCashier — import a broker report

Merges trades and coupons from a broker `.xls` report into the base GnuCash book.
Review the dry-run output before running the `execute` cell. Back up the book first.

Account mapping is read from `gnucashier.toml` (see `gnucashier.example.toml`).

In [ ]:
import os, shutil
from datetime import datetime, UTC
from pathlib import Path
os.environ["LD_LIBRARY_PATH"] = "/usr/lib/x86_64-linux-gnu/gnucash:" + os.environ.get("LD_LIBRARY_PATH", "")
from gnucashier.config import load_broker
from gnucashier.importer import BrokerImporter

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
base_file = 'books/Personal.gnucash'
# Pass the .zip directly, or a list of individual .xls files.
reports = ['imports/Брокерский (01.06.26-04.07.26).zip']
broker = load_broker('alfa')

In [ ]:
# Backup the base book before importing.
now = datetime.now(tz=UTC).strftime('%Y%m%dT%H%M%S')
backup = Path(base_file).with_name(Path(base_file).stem + f'-{now}.backup' + Path(base_file).suffix)
print(f'{base_file} -> {backup}')
shutil.copy(base_file, backup)

## Dry run

Aborts if commodities are missing ISINs — run `gnucashier backfill` first (or `importer.execute()` after `require_isins` handling). Optionally pre-check with `gnucashier validate ...`.

In [ ]:
importer = BrokerImporter(base_file, reports, broker)
importer.open_session()
importer.prepare()  # prints the dry-run analysis (incl. any missing-ISIN prerequisite)

## Import (only after reviewing the dry run)

In [ ]:
importer.execute()

In [ ]:
importer.close_session()